# Fast KMeans++ on MNIST

This tutorial compares FastKMeans++ with scikit-learn's KMeans on MNIST. It checks clustering quality first with adjusted Rand index, then compares runtime for `K=10` and `K=100`.

## Load MNIST

The data is scaled to `[0, 1]` before clustering. A fixed seed keeps centroid initialization reproducible.

In [ ]:
import time

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans as SklearnKMeans
from sklearn.datasets import fetch_openml
from sklearn.metrics import adjusted_rand_score

from fastkmeanspp import KMeans


# Fix the seed so both implementations use the same reproducible setup.
random_state = 42
mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X = np.asarray(mnist.data, dtype=np.float32) / 255
y = np.asarray(mnist.target, dtype=np.int8)
X.shape, y.shape

## Compare adjusted Rand index for `K=10`

ARI measures agreement with the known digit labels while correcting for chance. Higher values indicate better clustering alignment.

In [ ]:
def fit_model(model, X):
    start = time.perf_counter()
    model.fit(X)
    return model, time.perf_counter() - start


# Use one initialization and the same Lloyd iteration budget for both models.
sklearn_10 = SklearnKMeans(n_clusters=10, n_init=1, max_iter=20, random_state=random_state)
fast_10 = KMeans(n_clusters=10, n_iter=20, random_state=random_state)
sklearn_10.fit(X)
fast_10.fit(X)

ari_10 = pd.DataFrame(
    {"ARI": [
        adjusted_rand_score(y, sklearn_10.labels_),
        adjusted_rand_score(y, fast_10.labels_),
    ]},
    index=["scikit-learn", "fastkmeanspp"],
)
ari_10.style.format("{:.3f}")

## Compare speed for `K=10`

The first FastKMeans++ fit warms up the native distance kernel. The reported run measures a fresh fit after that warmup.

In [ ]:
# Warm up the native distance kernel before timing.
KMeans(n_clusters=10, n_iter=20, random_state=random_state).fit(X[:1000])
_, sklearn_time_10 = fit_model(SklearnKMeans(n_clusters=10, n_init=1, max_iter=20, random_state=random_state), X)
_, fast_time_10 = fit_model(KMeans(n_clusters=10, n_iter=20, random_state=random_state), X)

speed_10 = pd.DataFrame(
    {"fit time (s)": [sklearn_time_10, fast_time_10]},
    index=["scikit-learn", "fastkmeanspp"],
)
speed_10.style.format("{:.3f}")

## Compare adjusted Rand index for `K=100`

The second comparison repeats the ARI check with 100 clusters.

In [ ]:
# Fit both implementations before showing any timing results.
sklearn_100 = SklearnKMeans(n_clusters=100, n_init=1, max_iter=20, random_state=random_state)
fast_100 = KMeans(n_clusters=100, n_iter=20, random_state=random_state)
sklearn_100.fit(X)
fast_100.fit(X)

ari_100 = pd.DataFrame(
    {"ARI": [
        adjusted_rand_score(y, sklearn_100.labels_),
        adjusted_rand_score(y, fast_100.labels_),
    ]},
    index=["scikit-learn", "fastkmeanspp"],
)
ari_100.style.format("{:.3f}")

## Compare speed for `K=100`

These timings measure one fit of each implementation on the same MNIST matrix. Repeat the cells for a more stable machine-specific benchmark.

In [ ]:
# Warm up the larger candidate shape before measuring the native implementation.
KMeans(n_clusters=100, n_iter=20, random_state=random_state).fit(X[:1000])
_, sklearn_time_100 = fit_model(SklearnKMeans(n_clusters=100, n_init=1, max_iter=20, random_state=random_state), X)
_, fast_time_100 = fit_model(KMeans(n_clusters=100, n_iter=20, random_state=random_state), X)

speed_100 = pd.DataFrame(
    {"fit time (s)": [sklearn_time_100, fast_time_100]},
    index=["scikit-learn", "fastkmeanspp"],
)
speed_100.style.format("{:.3f}")